# Create Analysis Dataset

Maps reviewable CDM-lite source values to the shared vocabulary, enriches the mapped tables, and writes one hospital analysis file. Set `project_root` to the CDM project and `data_dir` to the output directory. The result is `data_dir/<hospital>/algo_doc_dataset.csv`.


## Setup Parameters
The following parameters can be used to allow running the notebooks using papermill for both hospitals, or modified directly in  the notebook to inspect manually.

Example:
```bash
poetry run papermill 1-create-analysis-dataset.ipynb vumc-out.ipynb -p hospital vumc
poetry run papermill 1-create-analysis-dataset.ipynb lumc-out.ipynb -p hospital lumc
```

In [ ]:
hospital = "vumc"  # "LUMC" | "VUmc"
project_root = "../"
data_dir = "../data"

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd

from algo_vs_doc import cohorts, enrichment, interim, value_mapping
from algo_vs_doc.scoring.concepts import LIFE_SUPPORT_CONCEPTS

hospital = cohorts.canonical_hospital(hospital)

pd.set_option("display.max_columns", None)

## Load the intermediate tables

Load the shared CDM-lite tables defined in `algo_vs_doc/scoring/cdm.py`. The optional `elective_surgery` table adjusts day-0 SOFA Neurology handling.


In [ ]:
tables = enrichment.load_cdm(project_root, hospital)

pd.DataFrame(
    [
        {"table": name, "rows": len(frame), "columns": frame.shape[1]}
        for name, frame in tables.items()
    ]
).set_index("table")

## Validate and map the contract

Require the shared stay columns, then strictly map reviewable non-proprietary values before enrichment. Unknown non-null source values stop the pipeline before an analysis CSV is written.


In [ ]:
enrichment.check_stay_columns(tables["icu_stays"])
print(
    f"{hospital}: {len(tables['icu_stays']):,} ICU stays, all {len(enrichment.STAY_COLUMNS)} required columns present"
)

tables = value_mapping.map_cdm_tables(tables, hospital=hospital)
print(tables["icu_stays"][["sex", "specialty"]].value_counts(dropna=False))

mapped_columns = [
    column
    for column in enrichment.MAPPED_STAY_COLUMNS
    if column in tables["icu_stays"].columns
]
print(f"canonical columns added by value mapping: {mapped_columns}")

unexpected = sorted(
    set(tables["icu_stays"].columns)
    - set(enrichment.STAY_COLUMNS)
    - set(enrichment.MAPPED_STAY_COLUMNS)
)
if unexpected:
    print(f"additional columns carried by the ETL: {unexpected}")

## Enrich

`enrichment.enrich` adds severity scores, condition and life-support, and prediction
columns. It retains all stays during the study period; Cohort Enrolment applies exclusions and compares groups.

In [ ]:
enriched = enrichment.enrich(tables, hospital=hospital)

study_rows = enriched["is_study_cohort"].astype(bool)
print(
    f"{int(study_rows.sum()):,} study-cohort stays and "
    f"{int((~study_rows).sum()):,} follow-up context stays enriched"
)
recorded_readmission = (
    enriched["has_icu_readmission_7days"]
    | enriched["has_elective_icu_readmission_7days"]
)
unlinked_readmission = (
    study_rows & recorded_readmission & ~enriched["readmission_stay_linked"]
)
print(
    "study-cohort seven-day ICU readmissions without a linked target stay:",
    int(unlinked_readmission.sum()),
)
enriched = enriched.loc[study_rows].copy()
print(f"{len(enriched):,} study-cohort stays x {enriched.shape[1]} columns")
print(f"with a physician prediction: {enriched['doc_predicted_at'].notna().sum():,}")
print(
    f"with a model prediction at discharge: {enriched['algo_prediction_at_discharge'].notna().sum():,}"
)
enriched.head()

### Severity and life-support summary

Inspect derived values before writing the dataset.


In [ ]:
display(enriched[list(enrichment.pipeline.SEVERITY_COLUMNS)].describe().T)

support_and_conditions = sorted(
    LIFE_SUPPORT_CONCEPTS | {"ventilation"} | enrichment.CONDITION_CONCEPTS
)
(
    enriched[support_and_conditions]
    .sum()
    .rename("stays")
    .to_frame()
    .assign(percentage=lambda f: (100 * f["stays"] / len(enriched)).round(1))
    .sort_values("stays", ascending=False)
)

## Save enriched dataset

Sort the enriched dataset deterministically, retain the local Parquet checkpoint,
and write the fully mapped `data_dir/<hospital>/algo_doc_dataset.csv`.

In [ ]:
dataset = (
    enriched.drop(columns=["specialty_id"])
    .sort_values(["icu_admitted_at", "patient_id", "icu_stay_id"])
    .reset_index(drop=True)
)

parquet_path = interim.interim_path(project_root, hospital, "enriched.parquet")
os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
dataset.to_parquet(parquet_path, index=False)

csv_path = cohorts.dataset_path(data_dir, hospital)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
dataset.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Wrote {len(dataset):,} stays to:")
print(" ", parquet_path)
print(" ", csv_path)
dataset.head()